#### Master CSV se DataFrame load 

In [33]:
import pandas as pd

df = pd.read_csv("../data_processed/all_stocks_daily.csv")
df['date'] = pd.to_datetime(df['date'])

df = df.sort_values(['ticker', 'date'])
df.head(), df.shape


(     ticker    close                date     high      low    month     open  \
 0  ADANIENT  2387.25 2023-10-03 05:30:00  2424.90  2372.00  2023-10  2418.00   
 1  ADANIENT  2464.95 2023-10-04 05:30:00  2502.75  2392.25  2023-10  2402.20   
 2  ADANIENT  2466.35 2023-10-05 05:30:00  2486.50  2446.40  2023-10  2477.95   
 3  ADANIENT  2478.10 2023-10-06 05:30:00  2514.95  2466.05  2023-10  2466.35   
 4  ADANIENT  2442.60 2023-10-09 05:30:00  2459.70  2411.30  2023-10  2440.00   
 
     volume  
 0  2019899  
 1  2857377  
 2  1132455  
 3  1510035  
 4  1408224  ,
 (14200, 8))

####  Daily return + Yearly return per stock

In [34]:
# daily return per ticker
df['daily_return'] = df.groupby('ticker')['close'].pct_change()

df[['ticker', 'date', 'close', 'daily_return']].head()


,ticker,date,close,daily_return
0,ADANIENT,2023-10-03 05:30:00,2387.25,NaN
1,ADANIENT,2023-10-04 05:30:00,2464.95,0.032548
2,ADANIENT,2023-10-05 05:30:00,2466.35,0.000568
3,ADANIENT,2023-10-06 05:30:00,2478.10,0.004764
4,ADANIENT,2023-10-09 05:30:00,2442.60,-0.014325


In [35]:
# first and last close per ticker
first_last = df.groupby('ticker').agg(
    first_close=('close', 'first'),
    last_close=('close', 'last')
).reset_index()

first_last['yearly_return'] = (first_last['last_close'] - first_last['first_close']) / first_last['first_close']

first_last.sort_values('yearly_return', ascending=False).head(10)


,ticker,first_close,last_close,yearly_return
47,TRENT,2059.10,6652.80,2.230926
8,BEL,139.20,280.85,1.017601
30,M&M,1537.40,3012.95,0.959770
5,BAJAJ-AUTO,5016.45,9481.65,0.890112
9,BHARTIARTL,925.30,1569.30,0.695990
35,POWERGRID,199.55,336.95,0.688549
10,BPCL,170.68,285.85,0.674772
20,HEROMOTOCO,3015.60,4794.10,0.589767
40,SUNPHARMA,1141.45,1795.30,0.572824
17,HCLTECH,1238.70,1898.40,0.532574


#### Top 10 green, top 10 loss stocks

In [36]:
top_10_green = first_last.sort_values('yearly_return', ascending=False).head(10)
top_10_red   = first_last.sort_values('yearly_return').head(10)

top_10_green, top_10_red


(        ticker  first_close  last_close  yearly_return
 47       TRENT      2059.10     6652.80       2.230926
 8          BEL       139.20      280.85       1.017601
 30         M&M      1537.40     3012.95       0.959770
 5   BAJAJ-AUTO      5016.45     9481.65       0.890112
 9   BHARTIARTL       925.30     1569.30       0.695990
 35   POWERGRID       199.55      336.95       0.688549
 10        BPCL       170.68      285.85       0.674772
 20  HEROMOTOCO      3015.60     4794.10       0.589767
 40   SUNPHARMA      1141.45     1795.30       0.572824
 17     HCLTECH      1238.70     1898.40       0.532574,
         ticker  first_close  last_close  yearly_return
 24  INDUSINDBK      1435.40      998.20      -0.304584
 3   ASIANPAINT      3166.85     2472.20      -0.219350
 7   BAJFINANCE      7967.60     6683.95      -0.161109
 0     ADANIENT      2387.25     2228.00      -0.066709
 22  HINDUNILVR      2468.90     2445.25      -0.009579
 32   NESTLEIND      2231.52     2247.30       

#### Market summary (green vs red, avg price, avg volume)

In [37]:
# green vs red count
green_mask = first_last['yearly_return'] > 0
red_mask   = first_last['yearly_return'] <= 0

n_green = green_mask.sum()
n_red   = red_mask.sum()

avg_price = df['close'].mean()
avg_volume = df['volume'].mean()

summary = {
    "total_stocks": first_last.shape[0],
    "green_stocks": int(n_green),
    "red_stocks": int(n_red),
    "pct_green": round(n_green / first_last.shape[0] * 100, 2),
    "pct_red": round(n_red / first_last.shape[0] * 100, 2),
    "avg_close_price": round(avg_price, 2),
    "avg_volume": round(avg_volume, 2),
}

summary


{'total_stocks': 50,
 'green_stocks': 45,
 'red_stocks': 5,
 'pct_green': np.float64(90.0),
 'pct_red': np.float64(10.0),
 'avg_close_price': np.float64(2449.42),
 'avg_volume': np.float64(6833474.65)}

####  Volatility (Top 10 most volatile stocks)


In [38]:
# daily_return already hai df me
volatility = (
    df.groupby('ticker')['daily_return']
      .std()
      .reset_index()
      .rename(columns={'daily_return': 'volatility'})
      .sort_values('volatility', ascending=False)
)

top_10_volatile = volatility.head(10)
top_10_volatile


,ticker,volatility
0,ADANIENT,0.028601
1,ADANIPORTS,0.026029
8,BEL,0.023283
47,TRENT,0.023074
34,ONGC,0.022247
10,BPCL,0.022069
39,SHRIRAMFIN,0.021687
13,COALINDIA,0.021411
21,HINDALCO,0.019587
33,NTPC,0.019475


#### Cumulative return (Top 5 performing stocks over time)

In [39]:
top_5_tickers = (
    first_last.sort_values('yearly_return', ascending=False)
              .head(5)['ticker']
              .tolist()
)
top_5_tickers


['TRENT', 'BEL', 'M&M', 'BAJAJ-AUTO', 'BHARTIARTL']

In [40]:
df_top5 = df[df['ticker'].isin(top_5_tickers)].copy()

df_top5['daily_return'] = df_top5.groupby('ticker')['close'].pct_change()

# cumulative return: (1 + r1) * (1 + r2) ... - 1
df_top5['cumulative_return'] = (
    1 + df_top5['daily_return']
).groupby(df_top5['ticker']).cumprod() - 1

df_top5[['ticker', 'date', 'close', 'cumulative_return']].head(20)


,ticker,date,close,cumulative_return
1420,BAJAJ-AUTO,2023-10-03 05:30:00,5016.45,NaN
1421,BAJAJ-AUTO,2023-10-04 05:30:00,4918.60,-0.019506
1422,BAJAJ-AUTO,2023-10-05 05:30:00,5011.05,-0.001076
1423,BAJAJ-AUTO,2023-10-06 05:30:00,5014.60,-0.000369
1424,BAJAJ-AUTO,2023-10-09 05:30:00,5007.30,-0.001824
1425,BAJAJ-AUTO,2023-10-10 05:30:00,5037.50,0.004196
1426,BAJAJ-AUTO,2023-10-11 05:30:00,5064.40,0.009559
1427,BAJAJ-AUTO,2023-10-12 05:30:00,5106.60,0.017971
1428,BAJAJ-AUTO,2023-10-13 05:30:00,5052.40,0.007166
1429,BAJAJ-AUTO,2023-10-16 05:30:00,5076.70,0.012010


####  Sector-wise performance (join sector CSV)

In [41]:
sector_df = pd.read_csv("../data_raw/Sector_data - Sheet1.csv")
sector_df.head()


,COMPANY,sector,Symbol
0,ADANI ENTERPRISES,MISCELLANEOUS,ADANI ENTERPRISES: ADANIGREEN
1,ADANI PORTS & SEZ,MISCELLANEOUS,ADANI PORTS & SEZ: ADANIPORTS
2,APOLLO HOSPITALS,MISCELLANEOUS,APOLLO HOSPITALS: APOLLOHOSP
3,ASIAN PAINTS,PAINTS,ASIAN PAINTS: ASIANPAINT
4,AXIS BANK,BANKING,AXIS BANK: AXISBANK


In [42]:
import pandas as pd

sector_df = pd.read_csv("../data_raw/Sector_data - Sheet1.csv")
sector_df.head()
sector_df.columns


Index(['COMPANY', 'sector', 'Symbol'], dtype='str')

In [43]:
sector_df = pd.read_csv("../data_raw/Sector_data - Sheet1.csv")
print(sector_df.columns)


Index(['COMPANY', 'sector', 'Symbol'], dtype='str')


In [44]:
sector_df = sector_df.rename(columns={
    'Symbol': 'ticker',          # ya 'Ticker' jo bhi tumhare output me ho
    'Sector': 'sector',          # ya 'Industry' jo bhi ho
})

sector_df['ticker'] = sector_df['ticker'].str.strip()
df['ticker'] = df['ticker'].str.strip()

sector_df.head()


,COMPANY,sector,ticker
0,ADANI ENTERPRISES,MISCELLANEOUS,ADANI ENTERPRISES: ADANIGREEN
1,ADANI PORTS & SEZ,MISCELLANEOUS,ADANI PORTS & SEZ: ADANIPORTS
2,APOLLO HOSPITALS,MISCELLANEOUS,APOLLO HOSPITALS: APOLLOHOSP
3,ASIAN PAINTS,PAINTS,ASIAN PAINTS: ASIANPAINT
4,AXIS BANK,BANKING,AXIS BANK: AXISBANK


In [45]:
first_last_with_sector = (
    first_last.merge(sector_df[['ticker', 'sector']], on='ticker', how='left')
)

first_last_with_sector.head()


,ticker,first_close,last_close,yearly_return,sector
0,ADANIENT,2387.25,2228.00,-0.066709,NaN
1,ADANIPORTS,831.40,1136.75,0.367272,NaN
2,APOLLOHOSP,5118.95,6935.10,0.354790,NaN
3,ASIANPAINT,3166.85,2472.20,-0.219350,NaN
4,AXISBANK,1041.05,1142.40,0.097354,NaN


In [46]:
sector_perf = (
    first_last_with_sector.groupby('sector')['yearly_return']
    .mean()
    .reset_index()
    .sort_values('yearly_return', ascending=False)
)

sector_perf


,sector,yearly_return


#### Stock price correlation 

In [47]:
# pivot: rows = date, columns = ticker, values = close
price_pivot = df.pivot_table(
    index='date',
    columns='ticker',
    values='close'
)

corr_matrix = price_pivot.pct_change().corr()  # better to use returns
corr_matrix.head()


ticker,ADANIENT,ADANIPORTS,APOLLOHOSP,ASIANPAINT,AXISBANK,BAJAJ-AUTO,BAJAJFINSV,BAJFINANCE,BEL,BHARTIARTL,...,SUNPHARMA,TATACONSUM,TATAMOTORS,TATASTEEL,TCS,TECHM,TITAN,TRENT,ULTRACEMCO,WIPRO
ticker,,,,,,,,,,,,,,,,,,,,,
ADANIENT,1.000000,0.874233,0.136501,0.291583,0.297736,0.201871,0.356547,0.336031,0.522680,0.282859,...,0.166256,0.203847,0.324071,0.423563,0.115790,0.141677,0.283281,0.190412,0.361152,0.224003
ADANIPORTS,0.874233,1.000000,0.168902,0.274045,0.380514,0.204456,0.398041,0.398769,0.585672,0.316537,...,0.215856,0.220655,0.339241,0.466043,0.132423,0.148034,0.296148,0.201076,0.402489,0.238986
APOLLOHOSP,0.136501,0.168902,1.000000,0.258055,0.179381,0.223900,0.249392,0.223142,0.224594,0.236810,...,0.286227,0.096700,0.212357,0.254731,0.106846,0.169118,0.253361,0.089516,0.272951,0.223738
ASIANPAINT,0.291583,0.274045,0.258055,1.000000,0.126055,0.167650,0.248090,0.296922,0.156953,0.146953,...,0.202262,0.244278,0.105847,0.287050,0.166699,0.162959,0.307404,0.223094,0.280647,0.140642
AXISBANK,0.297736,0.380514,0.179381,0.126055,1.000000,0.198477,0.393683,0.354079,0.330512,0.238602,...,0.143991,0.139747,0.232145,0.370957,0.094669,0.136469,0.145931,0.199748,0.397024,0.207802


In [48]:
corr_matrix.to_csv("../data_processed/correlation_matrix.csv")


#### Top 5 gainers/losers per month

In [49]:
df['month'] = df['date'].dt.to_period('M').astype(str)
df[['date', 'month']].head()


,date,month
0,2023-10-03 05:30:00,2023-10
1,2023-10-04 05:30:00,2023-10
2,2023-10-05 05:30:00,2023-10
3,2023-10-06 05:30:00,2023-10
4,2023-10-09 05:30:00,2023-10


##### Monthly return per stock

In [50]:
# per ticker, per month: first close, last close
monthly_fl = (
    df.groupby(['ticker', 'month'])
      .agg(
          first_close=('close', 'first'),
          last_close=('close', 'last')
      )
      .reset_index()
)

monthly_fl['monthly_return'] = (
    (monthly_fl['last_close'] - monthly_fl['first_close']) 
    / monthly_fl['first_close']
)

monthly_fl.head()


,ticker,month,first_close,last_close,monthly_return
0,ADANIENT,2023-10,2387.25,2294.65,-0.038789
1,ADANIENT,2023-11,2217.30,2358.55,0.063704
2,ADANIENT,2023-12,2362.70,2848.95,0.205803
3,ADANIENT,2024-01,2917.20,3142.00,0.077060
4,ADANIENT,2024-02,3153.50,3285.40,0.041827


#### Top 5 gainers/losers from each month

In [51]:
top5_gainers_monthly = (
    monthly_fl.sort_values(['month', 'monthly_return'], ascending=[True, False])
    .groupby('month')
    .head(5)
)

top5_losers_monthly = (
    monthly_fl.sort_values(['month', 'monthly_return'], ascending=[True, True])
    .groupby('month')
    .head(5)
)

top5_gainers_monthly.head(20), top5_losers_monthly.head(20)


(         ticker    month  first_close  last_close  monthly_return
 448   NESTLEIND  2023-10      2231.52     2423.48        0.086022
 182   COALINDIA  2023-10       291.90      314.25        0.076567
 70   BAJAJ-AUTO  2023-10      5016.45     5314.05        0.059325
 518     SBILIFE  2023-10      1292.45     1367.85        0.058339
 658       TRENT  2023-10      2059.10     2154.70        0.046428
 659       TRENT  2023-11      2197.65     2787.00        0.268173
 281  HEROMOTOCO  2023-11      3092.45     3819.05        0.234959
 141        BPCL  2023-11       178.45      217.85        0.220790
 211   EICHERMOT  2023-11      3282.25     3896.90        0.187265
 29   APOLLOHOSP  2023-11      4796.55     5528.95        0.152693
 114         BEL  2023-12       147.45      184.20        0.249237
 16   ADANIPORTS  2023-12       827.80     1024.35        0.237437
 2      ADANIENT  2023-12      2362.70     2848.95        0.205803
 296    HINDALCO  2023-12       517.20      614.85        0.18

#### Export all important tables for BI/SQL

In [52]:
out_dir = "../data_processed"
import os
os.makedirs(out_dir, exist_ok=True)

first_last.to_csv(f"{out_dir}/yearly_return_per_ticker.csv", index=False)
top_10_green.to_csv(f"{out_dir}/top10_green_stocks.csv", index=False)
top_10_red.to_csv(f"{out_dir}/top10_red_stocks.csv", index=False)
volatility.to_csv(f"{out_dir}/volatility_per_ticker.csv", index=False)
df_top5.to_csv(f"{out_dir}/top5_cumulative_return_timeseries.csv", index=False)
sector_perf.to_csv(f"{out_dir}/sector_performance.csv", index=False)
first_last_with_sector.to_csv(f"{out_dir}/yearly_return_with_sector.csv", index=False)
monthly_fl.to_csv(f"{out_dir}/monthly_return_per_ticker.csv", index=False)
top5_gainers_monthly.to_csv(f"{out_dir}/top5_gainers_per_month.csv", index=False)
top5_losers_monthly.to_csv(f"{out_dir}/top5_losers_per_month.csv", index=False)


In [53]:
volatility.to_csv("../data_processed/volatility_per_ticker.csv", index=False)
monthly_fl.to_csv("../data_processed/monthly_return_per_ticker.csv", index=False)
top5_gainers_monthly.to_csv("../data_processed/top5_gainers_per_month.csv", index=False)
top5_losers_monthly.to_csv("../data_processed/top5_losers_per_month.csv", index=False)
